# ALSRS — Model Evaluation (visual)

Interactive version of `evaluate_model.py`. Run the cells top to bottom.
Four experiments with tables and plots:

- **A. Weight comparison** — linear vs squared vs (1+déficit²).
- **B. Hyperparameter grid** — which config is best.
- **C. Learning curve** — does more data help?
- **D. Cross-validation** — robust metric (5-fold by point).

The model is **regression** (MAE/RMSE are the primary metrics); recall of
HIGH / NOT_SUITABLE is shown only to interpret the result in irrigation terms.


In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, train_test_split

# This notebook lives in ml/.
ML_DIR = Path.cwd() if Path.cwd().name == "ml" else Path.cwd() / "ml"
df = pd.read_csv(ML_DIR / "ml_dataset_cacao_ccn51.csv")

FEATURES = [
    "month", "biweek", "mean_C", "std_C",
    "precip_total_mm", "precip_rainy_days", "pet_mm",
    "spei_1m", "spei_3m", "spei_6m", "spei_12m",
    "AWC_mm", "Storage_mm", "P_acum_mm", "WRSI_1m", "deficit_1m", "oni",
]
TARGETS = ["future_deficit_1m", "future_deficit_3m", "future_deficit_6m"]

SEVERITY = {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "NOT_SUITABLE": 3}

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} cols")


In [ ]:
def cv_eval(df, target, weight_fn, model_params, n_splits=5):
    """5-fold by-point CV -> dict with model + 3 baselines (same folds).

    persist = same-window deficit immediately before t (honest, matched
    window size): deficit_1m for 1m, past_deficit_3m for 3m, past_deficit_6m
    for 6m. mean/median are constant baselines. R2 referential only."""
    persist_col = {
        "future_deficit_1m": "deficit_1m",
        "future_deficit_3m": "past_deficit_3m",
        "future_deficit_6m": "past_deficit_6m",
    }[target]
    gkf = GroupKFold(n_splits=n_splits)
    maes, rmses, r2s = [], [], []
    p_maes, p_rmses = [], []
    m_maes, m_rmses = [], []
    d_maes, d_rmses = [], []
    for tr_idx, va_idx in gkf.split(df, groups=df["point_id"]):
        m = RandomForestRegressor(random_state=42, n_jobs=-1, **model_params)
        Xtr, ytr = df.iloc[tr_idx][FEATURES], df.iloc[tr_idx][target]
        Xva, yva = df.iloc[va_idx][FEATURES], df.iloc[va_idx][target]
        m.fit(Xtr, ytr, sample_weight=weight_fn(ytr))
        pred = m.predict(Xva)
        maes.append(mean_absolute_error(yva, pred))
        rmses.append(rmse(yva, pred))
        r2s.append(r2_score(yva, pred))
        # same-window persistence (read from full df; filter NaN head rows)
        persist = df.iloc[va_idx][persist_col]
        valid = persist.notna() & yva.notna()
        p_maes.append(mean_absolute_error(yva[valid], persist[valid]))
        p_rmses.append(rmse(yva[valid], persist[valid]))
        # mean / median from the TRAIN fold
        mean_val = float(ytr.mean()); median_val = float(ytr.median())
        m_maes.append(mean_absolute_error(yva, np.full(len(yva), mean_val)))
        m_rmses.append(rmse(yva, np.full(len(yva), mean_val)))
        d_maes.append(mean_absolute_error(yva, np.full(len(yva), median_val)))
        d_rmses.append(rmse(yva, np.full(len(yva), median_val)))
    return {
        "mae": np.mean(maes), "rmse": np.mean(rmses), "r2": np.mean(r2s),
        "base_persist_mae": np.mean(p_maes), "base_persist_rmse": np.mean(p_rmses),
        "base_mean_mae": np.mean(m_maes), "base_mean_rmse": np.mean(m_rmses),
        "base_median_mae": np.mean(d_maes), "base_median_rmse": np.mean(d_rmses),
    }

print("Helpers ready.")


## Experiment A — Weight comparison

Which `sample_weight` works best? Primary metric MAE/RMSE; also recall of the
classes that justify irrigation (HIGH, NOT_SUITABLE).

In [ ]:
point_ids = df["point_id"].unique()
train_pts, test_pts = train_test_split(point_ids, test_size=0.20, random_state=42)
tr = df[df["point_id"].isin(train_pts)]
te = df[df["point_id"].isin(test_pts)]
Xtr = tr[FEATURES]
Xte = te[FEATURES]

reg_rows, rec_rows = [], []
for wname, wfn in WEIGHTS.items():
    preds = {}
    for target in TARGETS:
        m = make_model()
        m.fit(Xtr, tr[target], sample_weight=wfn(tr[target]))
        preds[target] = m.predict(Xte)
        reg_rows.append([wname, target,
                         mean_absolute_error(te[target], preds[target]),
                         rmse(te[target], preds[target])])

    true_cls = te[TARGETS].apply(lambda c: c.map(classify_deficit))
    pred_cls = pd.DataFrame({
        t: pd.Series(preds[t], index=te.index).map(classify_deficit)
        for t in TARGETS
    })
    true_sugg = true_cls.apply(lambda r: worst_of(*r), axis=1)
    pred_sugg = pred_cls.apply(lambda r: worst_of(*r), axis=1)
    for cls in ["HIGH", "NOT_SUITABLE"]:
        rec_rows.append([wname, cls, class_recall(true_sugg, pred_sugg, cls)])

reg_a = pd.DataFrame(reg_rows, columns=["weight", "target", "mae", "rmse"])
rec_a = pd.DataFrame(rec_rows, columns=["weight", "class", "recall"])

print("MAE / RMSE por peso y horizonte:")
print(reg_a.round(2).to_string(index=False))
print()
print("Recall de HIGH / NOT_SUITABLE por peso:")
print(rec_a.round(3).to_string(index=False))


In [ ]:
# ---- Plot A: MAE (grouped) + recall ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

reg_a.pivot(index="target", columns="weight", values="mae").plot(
    kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("MAE por peso y horizonte (menor = mejor)")
axes[0].set_ylabel("MAE (puntos % de déficit)")
axes[0].tick_params(axis="x", rotation=0)

rec_a.pivot(index="weight", columns="class", values="recall").plot(
    kind="bar", ax=axes[1], color=["#C44E52", "#8172B3"])
axes[1].set_title("Recall de HIGH y NOT_SUITABLE por peso")
axes[1].set_ylabel("recall")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=0)

fig.tight_layout()
plt.show()


## Experiment B — Hyperparameter grid

Small manual grid, cross-validated on the 6-month target. Shows whether
tuning helps at all (spoiler: barely).

In [ ]:
target = "future_deficit_6m"
grid = [
    dict(n_estimators=100, max_depth=None, min_samples_leaf=1),
    dict(n_estimators=100, max_depth=10,  min_samples_leaf=1),
    dict(n_estimators=100, max_depth=20,  min_samples_leaf=1),
    dict(n_estimators=300, max_depth=None, min_samples_leaf=1),
    dict(n_estimators=300, max_depth=10,  min_samples_leaf=1),
    dict(n_estimators=300, max_depth=20,  min_samples_leaf=1),
    dict(n_estimators=300, max_depth=None, min_samples_leaf=5),
    dict(n_estimators=500, max_depth=None, min_samples_leaf=1),
    dict(n_estimators=500, max_depth=10,  min_samples_leaf=1),
    dict(n_estimators=500, max_depth=20,  min_samples_leaf=1),
    dict(n_estimators=500, max_depth=10,  min_samples_leaf=5),
    dict(n_estimators=500, max_depth=20,  min_samples_leaf=5),
]

rows = []
for p in grid:
    mae, r = cv_eval(df, target, weight_linear, p)
    desc = f"n={p['n_estimators']} d={p['max_depth']} leaf={p['min_samples_leaf']}"
    rows.append([desc, mae, r])

tun_b = (pd.DataFrame(rows, columns=["params", "cv_mae", "cv_rmse"])
         .sort_values("cv_rmse").reset_index(drop=True))
print(tun_b.round(2).to_string(index=False))


In [ ]:
# ---- Plot B: horizontal bars (red = best) ----
tun_sorted = tun_b.sort_values("cv_rmse", ascending=True)
colors = ["#C44E52" if i == 0 else "#4C72B0" for i in range(len(tun_sorted))]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(tun_sorted["params"], tun_sorted["cv_rmse"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("CV RMSE (6m)")
ax.set_title("Hyperparameter grid (menor RMSE = mejor; rojo = mejor)")
plt.show()


## Experiment C — Learning curve

Train on 25 / 50 / 75 / 100 % of the train farms. If the error keeps dropping,
more data would help; if it flattens, we have enough.

In [ ]:
rows = []
for frac in [0.25, 0.50, 0.75, 1.0]:
    n = int(len(train_pts) * frac)
    subset = train_pts[:n]
    tr_sub = df[df["point_id"].isin(subset)]
    Xsub = tr_sub[FEATURES]
    for t in TARGETS:
        m = make_model()
        m.fit(Xsub, tr_sub[t], sample_weight=weight_linear(tr_sub[t]))
        pred = m.predict(Xte)
        rows.append([frac, len(subset), t,
                     mean_absolute_error(te[t], pred), rmse(te[t], pred)])

lc = pd.DataFrame(rows, columns=["frac", "n_farms", "target", "mae", "rmse"])
print("MAE por horizonte y numero de fincas:")
print(lc.pivot(index="n_farms", columns="target", values="mae").round(2).to_string())


In [ ]:
# ---- Plot C: learning curve (MAE and RMSE) ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for metric, ax in zip(["mae", "rmse"], axes):
    lc.pivot(index="n_farms", columns="target", values=metric).plot(
        marker="o", ax=ax)
    ax.set_title(f"Learning curve — {metric.upper()}")
    ax.set_xlabel("nº de fincas (train)")
    ax.set_ylabel(metric.upper())
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## Experiment D — Cross-validation (robust metric)

5-fold by point over all farms, comparing the model against **three baselines**
on the **same folds**:

- **persist** = same-window deficit immediately before t (honest: matched
  window size, so no dilution mismatch).
- **mean / median** = constant baselines from the train fold.

R2 is referential only (unreliable on zero-inflated targets).

In [ ]:
rows = []
for t in TARGETS:
    res = cv_eval(df, t, weight_linear, {})
    rows.append([t, res["mae"], res["rmse"], res["r2"],
                 res["base_persist_mae"], res["base_persist_rmse"],
                 res["base_mean_mae"], res["base_mean_rmse"],
                 res["base_median_mae"], res["base_median_rmse"]])

cv_d = pd.DataFrame(rows, columns=[
    "target", "cv_mae", "cv_rmse", "cv_r2",
    "persist_mae", "persist_rmse", "mean_mae", "mean_rmse",
    "median_mae", "median_rmse"])
print(cv_d.round(3).to_string(index=False))


In [ ]:
# ---- Plot D: model vs baselines (MAE and RMSE) ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(cv_d))
w = 0.2
for metric, ax in zip(["cv_mae", "cv_rmse"], axes):
    ax.bar(x - 1.5*w, cv_d[metric], w, label="modelo", color="#4C72B0")
    ax.bar(x - 0.5*w, cv_d["persist_" + metric[3:]], w, label="persist", color="#DD8452")
    ax.bar(x + 0.5*w, cv_d["mean_" + metric[3:]], w, label="mean", color="#55A868")
    ax.bar(x + 1.5*w, cv_d["median_" + metric[3:]], w, label="median", color="#8172B3")
    ax.set_xticks(x)
    ax.set_xticklabels(cv_d["target"])
    ax.set_ylabel("error (puntos % de déficit)")
    ax.set_title(metric + " — modelo vs baselines (menor = mejor)")
    ax.legend()
fig.tight_layout()
plt.show()


## Summary

- **Weight:** `linear` wins (best MAE and best recall of NOT_SUITABLE).
- **Tuning:** the default `n_estimators=300` is already optimal; `max_depth=10`
  hurts badly. The RF is robust to hyperparameters.
- **Learning curve:** error still drops a little from 143 -> 191 farms, but with
  diminishing returns — 240 farms is enough.
- **Robust metric (report this):** CV MAE ~10.6–11.2, RMSE ~12.8–13.8.
